In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import optuna

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [6]:
OVERSAMPLED_DATA_DIR = '../data/processed/smote_balanced.parquet'
ovr_sampled_df = pd.read_parquet(OVERSAMPLED_DATA_DIR)

UNDERSAMPLED_DATA_DIR = '../data/processed/moderate_undersampled.parquet'
undr_sampled_df = pd.read_parquet(UNDERSAMPLED_DATA_DIR)

In [7]:
# Covertype Mapping
covertype_class = {
        1: 'Spruce/Fir', 2: 'Lodgepole Pine',
        3: 'Ponderosa Pine', 4: 'Cottonwood/Willow',
        5: 'Aspen', 6: 'Douglas-fir', 7:'Krummholz'
}

ovr_sampled_df['class_name'] = ovr_sampled_df['Cover_Type'].map(covertype_class)
undr_sampled_df['class_name'] = undr_sampled_df['Cover_Type'].map(covertype_class)

ovr_sampled_class_counts = ovr_sampled_df['class_name'].value_counts().sort_values(ascending=True)
undr_sampled_class_counts = undr_sampled_df['class_name'].value_counts().sort_values(ascending=True)

In [8]:
ovr_sampled_df.head()

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,...,Soil_Type33,Soil_Type34,Soil_Type35,Soil_Type36,Soil_Type37,Soil_Type38,Soil_Type39,Soil_Type40,Cover_Type,class_name
0,2596,51,3,258,0,510,221,232,148,6279,...,0,0,0,0,0,0,0,0,5,Aspen
1,2590,56,2,212,-6,390,220,235,151,6225,...,0,0,0,0,0,0,0,0,5,Aspen
2,2804,139,9,268,65,3180,234,238,135,6121,...,0,0,0,0,0,0,0,0,2,Lodgepole Pine
3,2785,155,18,242,118,3090,238,238,122,6211,...,0,0,0,0,0,0,0,0,2,Lodgepole Pine
4,2595,45,2,153,-1,391,220,234,150,6172,...,0,0,0,0,0,0,0,0,5,Aspen


In [9]:
undr_sampled_df.head()

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,...,Soil_Type33,Soil_Type34,Soil_Type35,Soil_Type36,Soil_Type37,Soil_Type38,Soil_Type39,Soil_Type40,Cover_Type,class_name
0,3067,5,15,108,16,3691,200,210,146,1964,...,0,0,0,0,0,0,0,0,1,Spruce/Fir
1,3152,32,10,660,135,1727,218,218,134,2555,...,0,0,0,0,0,0,0,0,1,Spruce/Fir
2,2977,345,5,120,6,1718,209,231,160,3476,...,0,0,0,0,0,0,0,0,1,Spruce/Fir
3,3221,106,9,192,1,2768,235,229,125,872,...,0,0,0,0,0,0,0,0,1,Spruce/Fir
4,3194,4,15,499,67,2255,200,211,147,3404,...,0,0,0,0,0,0,0,0,1,Spruce/Fir


# Model Comparison & Hyperparameter Tuning
In this section we will compare the performance of various models using appropriate metrics:
- Baseline Classification Models like Logistic Regression, Decision Trees or SVC.
- Ensemble Learning Methods like Voting Classifier, Random Forest...

We will work hyperparameter tuning, describing how we searched for and selected the optimal parameters for each model.

## Baseline Classification Models
### Logistic Regression

In [10]:
from sklearn.linear_model import LogisticRegression

X = ovr_sampled_df.drop(columns=['Cover_Type', 'class_name'])
y = ovr_sampled_df['Cover_Type']
print(X.shape, y.shape)

lr = LogisticRegression()
lr.fit(X,y)

print(f"Predictions for first 5 OVR sampled data: {lr.predict(X[:5])}")

(1983107, 54) (1983107,)
Predictions for first 5 OVR sampled data: [2 2 2 2 2]


In [11]:
X_undr = undr_sampled_df.drop(columns=['Cover_Type', 'class_name'])
y_undr = undr_sampled_df['Cover_Type']

undr_sampled_lr = LogisticRegression()
undr_sampled_lr.fit(X_undr, y_undr)

print(f"Predictions for first 5 UDR sampled data: {undr_sampled_lr.predict(X_undr[:5])}")

Predictions for first 5 UDR sampled data: [1 1 2 1 1]


Over sampled data seems to over estimate the majoity class from the riginal distributions, so we will work on the under sampled datasets

---- Hay que hacer algo mas para demostrar porque sacamos esa conclusion, yo lo he puesto por las 5 predicciones esas solo y por elegir que dataset usar -----

In [14]:
def objective(trial):
    solver = trial.suggest_categorical("solver", ["lbfgs", "newton-cg", "saga"])

    # Penalty depends on solver compatibility.
    if solver == "saga":
        penalty = trial.suggest_categorical("penalty", ["l1", "l2"])
    else:
        penalty = "l2"

    C = trial.suggest_float("C", 1e-4, 1e2, log=True)
    max_iter = trial.suggest_int("max_iter", 500, 3000)

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            C=C,
            penalty=penalty,
            solver=solver,
            max_iter=max_iter,
            random_state=42,
            n_jobs=-1
        ))
    ])

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_undr, y_undr, cv=cv, scoring="f1_macro", n_jobs=-1)
    return scores.mean()

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("Best CV f1_macro:", study.best_value)
print("Best hyperparameters:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

best_params = study.best_params.copy()
best_logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        C=best_params["C"],
        penalty=best_params.get("penalty", "l2"),
        solver=best_params["solver"],
        max_iter=best_params["max_iter"],
        random_state=42,
        n_jobs=-1
    ))
])

best_logreg.fit(X_undr, y_undr)
print("Final model fitted with best hyperparameters.")